# 🤿 Notebook 2 — FathomNet: BLIP-2 Caption Inference

**Goal**: Load the FathomNet DB, run **BLIP-2** inference with *metadata-guided prompts*, and store captions in SQLite.

**Input**: Kaggle Dataset `fathomnet-underwater-db` (attach as input)
**Output**: `fathomnet_with_captions.db`

> **Kaggle environment**: Enable GPU (T4 16GB). Enable Internet.

## 0. Install & import

In [ ]:
!pip install transformers accelerate bitsandbytes Pillow tqdm pandas einops -q

In [ ]:
import os, sqlite3, shutil, time, hashlib
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

INPUT_DB  = Path('/kaggle/input/fathomnet-underwater-db/fathomnet.db')
INPUT_IMG = Path('/kaggle/input/fathomnet-underwater-db/images')

OUT_DIR   = Path('/kaggle/working/fathomnet_captioned')
OUT_DB    = OUT_DIR / 'fathomnet_with_captions.db'
OUT_CSV   = OUT_DIR / 'image_text_pairs.csv'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not OUT_DB.exists() and INPUT_DB.exists():
    shutil.copy2(INPUT_DB, OUT_DB)
    print(f'Copied DB -> {OUT_DB}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load BLIP-2 model (8-bit)

In [ ]:
MODEL_ID = 'Salesforce/blip2-opt-2.7b'
print(f'Loading {MODEL_ID} ...')

processor = Blip2Processor.from_pretrained(MODEL_ID)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
) if DEVICE == 'cuda' else None

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    quantization_config=quantization_config,
    device_map='auto' if DEVICE == 'cuda' else None,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
)
model.eval()
print('Model loaded!')

## 2. Prompt builder

In [ ]:
def build_prompt(row) -> str:
    species = str(getattr(row, 'species_clean', '') or '').strip()
    depth = getattr(row, 'depth_m', None)
    temp = getattr(row, 'temperature_c', None)
    
    parts = []
    if species: parts.append(f'showing {species}')
    if depth and not pd.isna(depth): parts.append(f'at {depth:.0f}m depth')
    if temp and not pd.isna(temp): parts.append(f'water temp {temp:.1f}C')
    
    ctx = ', '.join(parts)
    if ctx:
        return f'This is an underwater photo {ctx}. Describe what you see: '
    return 'Describe this underwater image: '

## 3. Inference loop

In [ ]:
conn = sqlite3.connect(OUT_DB)
df_images = pd.read_sql('SELECT * FROM images', conn)

# Restart safety: Get already processed pairs
try:
    done_ids = set(pd.read_sql('SELECT image_id FROM image_text_pairs', conn)['image_id'])
except Exception:
    done_ids = set()

df_todo = df_images[~df_images['image_id'].isin(done_ids)].copy()
print(f'Todo: {len(df_todo)}')

batch_size = 100
results = []

for i, row in enumerate(tqdm(df_todo.itertuples(), total=len(df_todo))):
    img_path = INPUT_IMG / row.filename
    if not img_path.exists():
        img_path = INPUT_IMG / 'api_images' / Path(row.filename).name
    
    if not img_path.exists(): continue
    
    try:
        image = Image.open(img_path).convert('RGB')
        prompt = build_prompt(row)
        
        inputs = processor(images=image, text=prompt, return_tensors='pt').to(DEVICE, torch.float16 if DEVICE == 'cuda' else torch.float32)
        
        with torch.no_grad():
            out = model.generate(
                **inputs, 
                max_new_tokens=60, 
                min_new_tokens=5, 
                repetition_penalty=1.5
            )
        
        caption = processor.batch_decode(out, skip_special_tokens=True)[0].strip()
        
        # OPT models append the input prompt to the output. We must remove it.
        clean_prompt = prompt.strip()
        if caption.startswith(clean_prompt):
            caption = caption[len(clean_prompt):].strip()
        
        results.append((row.image_id, caption, prompt, MODEL_ID))
        
        if len(results) >= batch_size:
            conn.executemany('INSERT INTO image_text_pairs (image_id, caption, prompt_used, model_name) VALUES (?,?,?,?)', results)
            conn.commit()
            results = []
    except Exception as e:
        print(f'Error {row.image_id}: {e}')

if results:
    conn.executemany('INSERT INTO image_text_pairs (image_id, caption, prompt_used, model_name) VALUES (?,?,?,?)', results)
    conn.commit()

conn.close()
print('Inference complete.')

## 4. Export CSV

In [ ]:
conn = sqlite3.connect(OUT_DB)
df_final = pd.read_sql('SELECT i.filename as image_path, p.caption FROM image_text_pairs p JOIN images i ON p.image_id = i.image_id', conn)
df_final.to_csv(OUT_CSV, index=False)
conn.close()
print(f'Saved CSV: {OUT_CSV}')